In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Total RAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB" if torch.cuda.is_available() else "")

# Or check CPU/system info:
import psutil
print(f"System RAM: {psutil.virtual_memory().total / 1e9:.2f} GB")
print(f"CPU cores: {psutil.cpu_count()}")

CUDA available: True
Number of GPUs: 1
GPU name: Tesla T4
Total RAM: 15.64 GB
System RAM: 13.61 GB
CPU cores: 2


In [2]:
!nvidia-smi -L


GPU 0: Tesla T4 (UUID: GPU-9d2ad689-68b8-0902-45af-3b92bb520555)


In [ ]:
!pip install -q bitsandbytes datasets accelerate loralib
!pip install -q git+https://github.com/huggingface/transformers.git@main git+https://github.com/huggingface/peft.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.4 MB/s eta 0:00:00:00:0100:01


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import torch
import torch.nn as nn
import bitsandbytes as bnb
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "bigscience/bloom-560m",
    quantization_config=bnb_config,
    device_map='auto',
)

tokenizer = AutoTokenizer.from_pretrained("bigscience/bloom-560m")

In [ ]:
print(f"Model loaded with {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")
print(model)

Model loaded with 559.21M parameters
BloomForCausalLM(
  (transformer): BloomModel(
    (word_embeddings): Embedding(250880, 1024)
    (word_embeddings_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (h): ModuleList(
      (0-23): 24 x BloomBlock(
        (input_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (self_attention): BloomAttention(
          (query_key_value): Linear8bitLt(in_features=1024, out_features=3072, bias=True)
          (dense): Linear8bitLt(in_features=1024, out_features=1024, bias=True)
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (post_attention_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): BloomMLP(
          (dense_h_to_4h): Linear8bitLt(in_features=1024, out_features=4096, bias=True)
          (gelu_impl): BloomGelu()
          (dense_4h_to_h): Linear8bitLt(in_features=4096, out_features=1024, bias=True)
        )
      )
    )
    (ln_f): L

In [ ]:
# freeze the model
for param in model.parameters():
    param.requires_grad = False
model.gradient_checkpointing_enable()
model.config.use_cache = False  # close the KV cache for gradient checkpointing
model.enable_input_require_grads()

class CastOutputToFloat(nn.Sequential):
    def forward(self, x):
        return super().forward(x).to(torch.float32)

model.lm_head = CastOutputToFloat(model.lm_head)

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
print_trainable_parameters(model)

# # Print all module names in your model
# for name, module in model.named_modules():
#     print(name)

trainable params: 0 || all params: 408219648 || trainable%: 0.0

transformer
transformer.word_embeddings
transformer.word_embeddings_layernorm
transformer.h
transformer.h.0
transformer.h.0.input_layernorm
transformer.h.0.self_attention
transformer.h.0.self_attention.query_key_value
transformer.h.0.self_attention.dense
transformer.h.0.self_attention.attention_dropout
transformer.h.0.post_attention_layernorm
transformer.h.0.mlp
transformer.h.0.mlp.dense_h_to_4h
transformer.h.0.mlp.gelu_impl
transformer.h.0.mlp.dense_4h_to_h
transformer.h.1
transformer.h.1.input_layernorm
transformer.h.1.self_attention
transformer.h.1.self_attention.query_key_value
transformer.h.1.self_attention.dense
transformer.h.1.self_attention.attention_dropout
transformer.h.1.post_attention_layernorm
transformer.h.1.mlp
transformer.h.1.mlp.dense_h_to_4h
transformer.h.1.mlp.gelu_impl
transformer.h.1.mlp.dense_4h_to_h
transformer.h.2
transformer.h.2.input_layernorm
transformer.h.2.self_attention
transformer.h.2.self_a

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16, #attention heads
    lora_alpha=32, #alpha scaling
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM" # set this for CLM or Seq2Seq
)

model = get_peft_model(model, config)
print_trainable_parameters(model)

trainable params: 1572864 || all params: 409792512 || trainable%: 0.3838196047857507


In [ ]:
# Data

import transformers
from datasets import load_dataset
data = load_dataset("Abirate/english_quotes")



In [ ]:
def merge_columns(example):
    example["prediction"] = example["quote"] + " ->: " + str(example["tags"])
    return example

data['train'] = data['train'].map(merge_columns)
data['train']["prediction"][:5]

["“Be yourself; everyone else is already taken.” ->: ['be-yourself', 'gilbert-perreira', 'honesty', 'inspirational', 'misattributed-oscar-wilde', 'quote-investigator']",
 "“I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.” ->: ['best', 'life', 'love', 'mistakes', 'out-of-control', 'truth', 'worst']",
 "“Two things are infinite: the universe and human stupidity; and I'm not sure about the universe.” ->: ['human-nature', 'humor', 'infinity', 'philosophy', 'science', 'stupidity', 'universe']",
 "“So many books, so little time.” ->: ['books', 'humor']",
 "“A room without books is like a body without a soul.” ->: ['books', 'simile', 'soul']"]

In [ ]:
len(data['train'])
data['train'][0]

{'quote': '“Be yourself; everyone else is already taken.”',
 'author': 'Oscar Wilde',
 'tags': ['be-yourself',
  'gilbert-perreira',
  'honesty',
  'inspirational',
  'misattributed-oscar-wilde',
  'quote-investigator'],
 'prediction': "“Be yourself; everyone else is already taken.” ->: ['be-yourself', 'gilbert-perreira', 'honesty', 'inspirational', 'misattributed-oscar-wilde', 'quote-investigator']"}

In [ ]:
data = data.map(lambda samples: tokenizer(samples['prediction']), batched=True)

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [ ]:
data['train'][0]['prediction']

"“Be yourself; everyone else is already taken.” ->: ['be-yourself', 'gilbert-perreira', 'honesty', 'inspirational', 'misattributed-oscar-wilde', 'quote-investigator']"

In [ ]:
print(data['train'][0].keys())

dict_keys(['quote', 'author', 'tags', 'prediction', 'input_ids', 'attention_mask'])


In [ ]:
#Training

from peft import LoraConfig, get_peft_model, PeftModel

# Apply LoRA if not already applied (e.g. after a kernel restart)
if not isinstance(model, PeftModel):
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    print_trainable_parameters(model)

trainer = transformers.Trainer(
    model=model, 
    train_dataset=data['train'],
    args=transformers.TrainingArguments(
        per_device_train_batch_size=4, 
        gradient_accumulation_steps=4,
        warmup_steps=100, 
        max_steps=2, 
        learning_rate=2e-4, 
        fp16=False,  # disable fp16 to avoid NaN with 8-bit
        logging_steps=1, 
        output_dir='outputs'
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
trainer.train()

tensor(nan, device='cuda:0') tensor(False, device='cuda:0')


In [ ]:
def tokenize(batch):
    return tokenizer(batch["prediction"], truncation=True, max_length=128)

text_cols = ["quote", "author", "tags", "prediction"]
data = data.map(tokenize, batched=True, remove_columns=[c for c in text_cols if c in data['train'].column_names])

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [ ]:
# debug cell removed - all checks completed previously


--- Dataset summary ---
type(data): <class 'datasets.dataset_dict.DatasetDict'>
splits: ['train']
train columns: ['input_ids', 'attention_mask']
example keys: ['input_ids', 'attention_mask']
input_ids length: 53
decoded: “Be yourself; everyone else is already taken.” ->: ['be-yourself', 'gilbert-perreira', 'honesty', 'inspirational', 'misattributed-oscar-wilde', 'quote-investigator']

--- Collated batch info ---
input_ids <class 'torch.Tensor'> torch.Size([2, 84]) torch.int64
attention_mask <class 'torch.Tensor'> torch.Size([2, 84]) torch.int64
labels <class 'torch.Tensor'> torch.Size([2, 84]) torch.int64
labels -100 count: 31
labels sample (first row): tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  1502, 17143, 33218,    30, 39839,  4384,   632, 11226, 15713,
           17,   982, 1195

In [ ]:
# fp32 and quant test removed



--- testing fp32 model on same batch ---


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

fp32 loss: tensor(nan) logits finite? tensor(False)
--- end fp32 test ---


In [ ]:
# input/embedding diagnostics removed


input_ids min/max: tensor(3, device='cuda:0') tensor(239002, device='cuda:0')
input_ids sample[0]: tensor([    3,     3,     3,     3,     3,     3,     3,     3,     3,     3,
            3,     3,     3,     3,     3,     3,     3,     3,     3,     3,
            3,     3,     3,     3,     3,     3,     3,     3,     3,     3,
            3,  1502, 17143, 33218,    30, 39839,  4384,   632, 11226, 15713,
           17,   982, 11953,    29, 24629,  2765, 17731,  3240, 15407,    10,
           15, 83077,   354, 26624, 31683, 71421,    10,    15,   756, 19218,
        56452,    10,    15,   756, 71538,  3383,    10,    15, 29412,   290,
        96783, 11914, 43555,  5231, 16728, 51464,    10,    15,   756, 67091,
        15595, 51261,  2623,  3166], device='cuda:0')
simple input_ids: tensor([[101579,   8876]])
simple logits finite? tensor(True)
simple loss: None
logits min/max/nan count/inf count: tensor(nan, dtype=torch.float16, grad_fn=<MinBackward1>) tensor(nan, dtype=torch.float16,

In [ ]:
# embedding row check removed


any NaN in emb matrix? tensor(False)
any Inf in emb matrix? tensor(False)
row 239002 any NaN? tensor(False) any Inf? tensor(False)
row sample: tensor([ 0.0202, -0.0054,  0.0119, -0.0051, -0.0090,  0.0006, -0.0047,  0.0089,
        -0.0200,  0.0019], dtype=torch.float16, grad_fn=<SliceBackward0>)


In [ ]:
# dtype check removed


transformer.word_embeddings.weight torch.float16


In [ ]:
# final fp32 reload removed; training will use existing model above


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

param dtype: torch.float32
loss: tensor(3.6060) finite? tensor(True)
